In [ ]:
from pathlib import Path

import prism

from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    GenericMaterials,
    GenericStocks,
    MaterialIntensities,
    RestOf, 
    SharesInflowStocks
)
from imagematerials.preprocessing import get_preprocessing_data

from imagematerials.rest_of import rest_of_preprocessing

from imagematerials.electricity.preprocessing import (
    get_preprocessing_data_gen,
    get_preprocessing_data_grid,
    get_preprocessing_data_stor
)

In [ ]:
YEAR_START = 1971  # start year of the simulation period
YEAR_END = 2100    # end year of the calculations
YEAR_OUT = 2100    # year of output generation = last year of reporting

In [ ]:
scenario_list = {
    "SSP2_M_CP":("SSP2_M_CP", None)
                 }

    # "SSP2_VLLO":("SSP2_VLLO", None),
    # "SSP2_VLLO_LifeTech":("SSP2_VLLO_LifeTech", ["resource_efficient"]

In [ ]:
path_image_scenarios = Path("..", "data", "raw", "image")
scenario_base_path = Path("..", "data", "raw", "circular_economy_scenarios")
path_raw_data = Path("..", "data", "raw")
path_image_materials = Path("..")

In [ ]:
model_runs = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    print(f"Running scenario: {scen_id}")
    print(f"Circular economy scenario: {circular_scen}")
    print('Current path: ', Path.cwd())

    # needed for electricity
    scen = climate_scen.split("_")[0]
    variant = "_".join(climate_scen.split("_")[1:])
    
    climate_policy_scenario_dir = path_image_scenarios / climate_scen
    if circular_scen is not None:
        circular_economy_dir = {
            scenario: scenario_base_path / scenario for scenario in circular_scen
        }
    else:
        circular_economy_dir = None

    # Define the complete timeline, including historic tail
    time_start = 1971
    complete_timeline = prism.Timeline(time_start, 2100, 1)
    simulation_timeline = prism.Timeline(1971, 2100, 1)

    # BUILDINGS
    bld_sector = get_preprocessing_data("buildings", path_raw_data, 
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dirs = circular_economy_dir) 
    # VEHICLES
    vhc_sector = get_preprocessing_data("vehicles", path_raw_data, 
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dirs = circular_economy_dir)
    prep_data_vhc = vhc_sector.prep_data
    vhc_sector = Sector('vehicles', prep_data_vhc)

    # GENERATION
    prep_data_gen = get_preprocessing_data_gen(path_image_materials, scen, variant, YEAR_START, YEAR_END, YEAR_OUT)
    sec_electr_gen = Sector("generation", prep_data_gen)
    
    # GRID
    prep_data_lines, prep_data_add = get_preprocessing_data_grid(path_image_materials, scen, variant, YEAR_START, YEAR_END, YEAR_OUT)
    sec_electr_grid_lines = Sector("grid", prep_data_lines)
    sec_electr_grid_add = Sector("grid_additional", prep_data_add)
   
    # STORAGE
    prep_data_phs, prep_data_oth_storage = get_preprocessing_data_stor(path_image_materials, scen, variant, YEAR_START, YEAR_END, YEAR_OUT)
    sec_electr_stor_phs = Sector("storage_pumped_hydropower", prep_data_phs)
    sec_electr_stor_oth = Sector("storage_other", prep_data_oth_storage)

    # REST OF SECTOR
    rest_sector = rest_of_preprocessing(path_raw_data, 
                        image_scenario_directory = climate_policy_scenario_dir, 
                        scenario = climate_scen)
    rest_sector = Sector(name='rest_of', data = rest_sector)
    

    factory = ModelFactory(
    [bld_sector, vhc_sector, sec_electr_gen, rest_sector, 
     sec_electr_grid_lines, sec_electr_grid_add, sec_electr_stor_phs, sec_electr_stor_oth], complete_timeline #
    ).add(GenericStocks, ["buildings", "vehicles", "generation", "grid", "grid_additional", 'storage_pumped_hydropower'] # 
    ).add(GenericMaterials,  "vehicles"
    ).add(SharesInflowStocks, "storage_other"
    ).add(MaterialIntensities, ["buildings", "generation", "grid", 
                                "grid_additional", "storage_pumped_hydropower", "storage_other"] # 
                                
    ).add(RestOf, "rest_of", input_sources={
        "gompertz_coefs": "rest_of",
        "gdp_per_capita": "rest_of",
        "population": "rest_of",
        "historic_diff_consumption_mean": "rest_of",
        "historic_diff_consumption_total": "rest_of"
}
    )
    model = factory.finish()
    model_runs[climate_scen] = model

    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        model.simulate(simulation_timeline)
        model.save_pkl(f'model_results/{scen_id}_model.pkl')



In [ ]:
from imagematerials.rest_of.util import sum_inflows_for_all_sectors, save_sum_as_csv

base_run = ModelFactory.load_pkl(f"model_results/SSP2_M_CP_model.pkl")

list_sum_sectors = ["buildings", "vehicles", "generation", "grid"]
list_sum_sectors_all = ["buildings", "vehicles", "generation", "grid", "grid_additional", "storage_pumped_hydropower", "storage_other"]

# total_inflow_1 = sum_inflows_for_all_sectors(base_run, 'inflow_materials', list_sum_sectors)
total_inflow = sum_inflows_for_all_sectors(base_run, 'inflow_materials', list_sum_sectors_all)


In [ ]:
# ((total_inflow_2.sel(material = "aluminium").sum() - total_inflow_1.sel(material = "aluminium").sum()) / total_inflow_2.sel(material = "aluminium").sum())*100


In [ ]:
numeric_region_map = {
    "CAN" : 'class_ 1',
    "USA": 'class_ 2',
    "MEX": 'class_ 3',
    "RCAM": 'class_ 4',
    "BRA": 'class_ 5',
    "RSAM": 'class_ 6',
    "NAF": 'class_ 7',
    "WAF": 'class_ 8',
    "EAF": 'class_ 9',
    "SAF": 'class_ 10',
    "WEU": 'class_ 11',
    "CEU": 'class_ 12',
    "TUR": 'class_ 13',
    "UKR": 'class_ 14',
    "STAN": 'class_ 15',
    "RUS": 'class_ 16',
    "ME": 'class_ 17',
    "INDIA": 'class_ 18',
    "KOR": 'class_ 19',
    "CHN": 'class_ 20',
    "SEAS": 'class_ 21',
    "INDO": 'class_ 22',
    "JAP": 'class_ 23',
    "OCE": 'class_ 24',
    "RSAS": 'class_ 25',
    "RSAF": 'class_ 26'
    }


In [ ]:
# detect Region coord name (case variations) and apply mapping
dim = "Region" if "Region" in total_inflow.coords else "region"
old_regions = [str(r) for r in total_inflow.coords[dim].values]

# ensure mapping keys are strings
map_str = {str(k): v for k, v in numeric_region_map.items()}
new_labels = [map_str.get(r, r) for r in old_regions]

# assign new labels, aggregate duplicates, then ensure class_ 1..class_26 exist
total_inflow = total_inflow.assign_coords({dim: ("Region", new_labels)})  # keep dim name consistent
total_inflow = total_inflow.groupby(dim).sum()

desired = [f"class_ {i}" for i in range(1, 27)]
total_inflow = total_inflow.reindex({dim: desired}, fill_value=0)

In [ ]:
save_sum_as_csv(total_inflow, "steel")
save_sum_as_csv(total_inflow, "aluminium")
save_sum_as_csv(total_inflow, "copper")

In [ ]:
# plot steel inflow stacked by Type (preserves Type dimension) with unique colors
df = total_inflow.sel(material='copper').sum('Region').loc[1972:].fillna(0)
df = df.pint.to('Mt')
df = df.to_pandas()


# ensure predictable column order and string labels
df = df.astype(float)
df.columns = [str(c) for c in df.columns]
df = df.reindex(columns=sorted(df.columns))

import matplotlib.pyplot as plt
import numpy as np

ncols = len(df.columns)

# build a palette: try concatenating tab20 / tab20b / tab20c, else sample a continuous map
try:
    tab20 = plt.cm.get_cmap('tab20')(np.linspace(0, 1, 20))
    tab20b = plt.cm.get_cmap('tab20b')(np.linspace(0, 1, 20))
    tab20c = plt.cm.get_cmap('tab20c')(np.linspace(0, 1, 20))
    base_colors = np.vstack([tab20, tab20b, tab20c])
except Exception:
    # fallback if some maps not available
    base_colors = plt.cm.get_cmap('tab20')(np.linspace(0, 1, max(20, ncols)))

if ncols <= base_colors.shape[0]:
    colors = [tuple(c) for c in base_colors[:ncols]]
else:
    # if still not enough, sample a continuous hue map (hsv) for distinct colors
    cmap = plt.cm.get_cmap('hsv')
    colors = [cmap(i / ncols) for i in range(ncols)]

fig, ax = plt.subplots(figsize=(10, 6))
df.plot.area(ax=ax, stacked=True, color=colors)
ax.set_ylabel('Inflow')
ax.set_title('Steel inflow stacked by Type')
ax.legend(title='Type', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()